# Logit Lens Analysis

Purpose: Find where in the model the failure happens layer by layer.

- At which layer does the model "commit" to the wrong action token?
- Do early layers already encode the wrong prediction, or does it happen late?
- Compare layer-by-layer predictions: success cases vs failure cases
> Key question: Is the repetition counting error introduced early or late in the network?

## 1. Preparation

In [88]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import torch

import sys, os
root_path = os.path.abspath("..")
src_path = os.path.join(root_path, "src")
for path in [root_path, src_path]:
    if path not in sys.path:
        sys.path.append(root_path)

from config.config import Config
from src.data import SCANDataModule,SCANTokenizer
from src.model import Transformer

In [ ]:
cfg = Config()
model = Transformer(cfg)
model.load_state_dict(torch.load("../results/model_final_small.pt"))

<All keys matched successfully>

In [90]:
import re
import pandas as pd

def parse_failure_log(filepath):
    """
    Parses SCAN failure logs and returns a list of dictionaries 
    with columns: splitname, command, target, pred.
    """
    # Regex breakdown:
    # \[(.*?)\] -> Captures everything inside the square brackets (splitname)
    # COMMAND:.*?IN:\s*(.*?)\s*OUT: -> Captures everything between IN: and OUT: (command)
    # TARGET:\s*(.*?)\s*\| -> Captures everything between TARGET: and the next | (target)
    # PRED:\s*(.*)$ -> Captures everything after PRED: to the end of the line (pred)
    pattern = re.compile(r"\[(.*?)\]\s*COMMAND:\s*(.*?)\s*\|\s*TARGET:\s*(.*?)\s*\|\s*PRED:\s*(.*)$")
    
    parsed_rows = []
    
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # Skip empty lines
                
            match = pattern.search(line)
            if match:
                splitname, command, target, pred = match.groups()
                
                parsed_rows.append({
                    "splitname": splitname,
                    "command": command.strip(),
                    "target": target.strip(),
                    "pred": pred.strip()
                })
                
    return parsed_rows

In [91]:
simple_data = parse_failure_log("../results/small_failure_cases_simple.txt")
length_data = parse_failure_log("../results/small_failure_cases_length.txt")
addprim_jump_data = parse_failure_log("../results/small_failure_cases_addprim_jump.txt")

all_failures = pd.DataFrame(simple_data + length_data + addprim_jump_data)
simple_failures = pd.DataFrame(simple_data)
length_failures = pd.DataFrame(length_data)
addprim_jump_failures = pd.DataFrame(addprim_jump_data)

simple_failures.head()

,splitname,command,target,pred
0,simple,<sos> IN: turn opposite right thrice and turn ...,I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_...,I_TURN_RIGHT I_TURN_RIGHT I_TURN_RIGHT I_TURN_...
1,simple,<sos> IN: look around right twice and turn lef...,I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN...,I_TURN_RIGHT I_LOOK I_TURN_RIGHT I_LOOK I_TURN...
2,simple,<sos> IN: jump around left thrice and run righ...,I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_L...,I_TURN_LEFT I_JUMP I_TURN_LEFT I_JUMP I_TURN_L...
3,simple,<sos> IN: jump opposite right after walk aroun...,I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN...,I_TURN_RIGHT I_WALK I_TURN_RIGHT I_WALK I_TURN...
4,simple,<sos> IN: look opposite right twice and jump o...,I_TURN_RIGHT I_TURN_RIGHT I_LOOK I_TURN_RIGHT ...,I_TURN_RIGHT I_TURN_RIGHT I_LOOK I_TURN_RIGHT ...


## 2. Interpret with TransformerLens

In [92]:
hooked_cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=4,
    d_model=128,       
    d_head=32,          
    d_mlp=512,        
    d_vocab= 25,
    n_ctx=128,          
    act_fn="gelu",      
    normalization_type="LN",
) 

In [93]:
tl_model = HookedTransformer(hooked_cfg)
# Load original custom weights
custom_state_dict = torch.load("../results/model_final_small.pt")

# Build a state dict for TransformerLens
tl_state_dict = {}

# Keep track of IGNORE keys
keys_to_skip = ["tfblocks.0.attn.IGNORE", "tfblocks.1.attn.IGNORE"]

for key, weight in custom_state_dict.items():
    if key in keys_to_skip:
        continue
        
    new_key = key
    
    # Map the layer block prefix: "tfblocks.0..." -> "blocks.0..."
    if new_key.startswith("tfblocks."):
        new_key = new_key.replace("tfblocks.", "blocks.")
        
    # Map the final layer norm: "ln.w" -> "ln_final.w"
    if new_key.startswith("ln."):
        new_key = new_key.replace("ln.", "ln_final.")
        
    # Map token embeddings: "embed.token_embed.weight" -> "embed.W_E"
    if new_key == "embed.token_embed.weight":
        new_key = "embed.W_E"
        
    # Map positional embeddings: "embed.pos_emb.weight" -> "pos_embed.W_pos"
    if new_key == "embed.pos_emb.weight":
        new_key = "pos_embed.W_pos"
        
    tl_state_dict[new_key] = weight

# Handle TransformerLens internal tracking values
# TransformerLens has a couple of default buffers (like mask strings) it initializes itself.
# We set strict=False so it doesn't crash over its own missing 'mask' or 'IGNORE' keys.
missing_keys, unexpected_keys = tl_model.load_state_dict(tl_state_dict, strict=False)

# Verify the load was clean
print("Missing keys:", [k for k in missing_keys if "mask" not in k and "IGNORE" not in k])
print("Unexpected keys:", unexpected_keys)

Missing keys: []
Unexpected keys: []


In [96]:
import torch
import pandas as pd

class LogitLensEvaluator:
    def __init__(self, tl_model, dm, cfg, max_new_tokens=128):
        """
        Initializes the evaluation engine.
        
        Args:
            tl_model: Loaded HookedTransformer instance
            dm: Initialized SCANDataModule instance
            cfg: Configuration object containing device specs
            max_new_tokens (int): Max steps to roll out generation
        """
        self.model = tl_model
        self.dm = dm
        self.cfg = cfg
        self.max_new_tokens = max_new_tokens
        self.tokenizer = dm.tokenizer
        
        # Internal storage
        self.df_failures = None
        self.generation_results = []
        self.split_name = ""

    def run_logit_lens(self, df, split_name="SCAN Split", sample_n=None, random_state=42):
        """
        Runs the dynamic logit lens loop over a provided DataFrame of failure cases.
        
        Args:
            df (pd.DataFrame): DataFrame containing 'command' and 'target' columns.
            split_name (str): Label for printing metrics later (e.g., 'Simple', 'Length').
            sample_n (int, optional): If provided, randomly samples N rows to run.
            random_state (int): Seed for reproducibility when sampling.
        """
        self.split_name = split_name
        self.generation_results = [] # Clear history from previous runs
        
        # Handle optional sampling
        if sample_n is not None and sample_n < len(df):
            df_to_run = df.sample(n=sample_n, random_state=random_state)
        else:
            df_to_run = df
            
        print(f"Running dynamic logit lens on {len(df_to_run)} examples from '{self.split_name}' split...")
        
        for _, row in df_to_run.iterrows():
            command = row["command"]
            target_str = row["target"]
            
            token_list = self.tokenizer.encode(command)
            token_ids = torch.tensor(token_list, dtype=torch.long).unsqueeze(0).to(self.cfg.device)
            target_tokens = self.tokenizer.encode(target_str) 
            
            history = []
            
            for step in range(self.max_new_tokens):
                logits, cache = self.model.run_with_cache(token_ids)
                
                l1_resid = cache["blocks.0.hook_resid_post"][:, -1, :]
                l2_resid = cache["blocks.1.hook_resid_post"][:, -1, :]
                
                l1_next_token = self.model.unembed(self.model.ln_final(l1_resid)).argmax(dim=-1).item()
                l2_next_token = self.model.unembed(self.model.ln_final(l2_resid)).argmax(dim=-1).item()
                
                if step < len(target_tokens):
                    true_token_str = self.tokenizer.decode([target_tokens[step]])
                else:
                    true_token_str = "<eos>"
                    
                l1_str = self.tokenizer.decode([l1_next_token])
                l2_str = self.tokenizer.decode([l2_next_token])
                
                if l2_str == true_token_str:
                    status = "Correct"
                elif l1_str == true_token_str and l2_str != true_token_str:
                    status = "Late Failure (Layer 2 Overwrite)"
                else:
                    status = "Early Failure (Layer 1 Dissolution)"
                    
                history.append({
                    "step": step,
                    "ground_truth": true_token_str,
                    "layer_1_choice": l1_str,
                    "layer_2_choice": l2_str,
                    "step_status": status
                })
                
                next_token_tensor = torch.tensor([[l2_next_token]], dtype=torch.long).to(self.cfg.device)
                token_ids = torch.cat([token_ids, next_token_tensor], dim=-1)
                
                if l2_next_token == 2 or l2_next_token == getattr(self.tokenizer, 'eos_token_id', None):
                    break
                    
            self.generation_results.append({
                "command": command,
                "target": target_str,
                "generation_trajectory": history
            })
            
        return self

    def print_metrics(self):
        """
        Aggregates data and prints structural research metrics.
        """
        if not self.generation_results:
            print("No analysis history found. Run .run_logit_lens() first.")
            return
            
        first_error_statuses = []
        error_steps = []
        
        for case in self.generation_results:
            trajectory = case["generation_trajectory"]
            
            for step_data in trajectory:
                if step_data["step_status"] != "Correct":
                    first_error_statuses.append(step_data["step_status"])
                    error_steps.append(step_data["step"])
                    break
                    
        status_counts = pd.Series(first_error_statuses).value_counts()
        status_percentages = pd.Series(first_error_statuses).value_counts(normalize=True) * 100
        
        print(f"\n=================== SCAN {self.split_name.upper()} SPLIT METRICS ===================")
        for status in status_counts.index:
            print(f"{status:<38}: {status_counts[status]:>4} cases ({status_percentages[status]:.1f}%)")
        print("-" * 65)
        if error_steps:
            print(f"Average generation step where failure begins: {pd.Series(error_steps).mean():.2f}")
        print("=================================================================\n")

    def inspect_sample(self, idx=0, only_late_failures=False):
        """
        Prints a single command's step-by-step trajectory in a clean table layout.
        """
        dataset = self.generation_results
        if only_late_failures:
            dataset = [c for c in self.generation_results if any(s["step_status"] == "Late Failure (Layer 2 Overwrite)" for s in c["generation_trajectory"])]
            
        if idx >= len(dataset):
            print(f"Index out of bounds. Available samples: {len(dataset)}")
            return
            
        # sample = dataset[idx]
        # print(f"\n[INSPECT] Command: {sample['command']}")
        # print(f"[INSPECT] Target:  {sample['target']}")
        # df_traj = pd.DataFrame(sample['generation_trajectory'])
        # print(df_traj.to_string(index=False))

In [98]:
dm = SCANDataModule(cfg)
tokenizer = dm.tokenizer

# 1. Initialize the evaluator once
evaluator = LogitLensEvaluator(tl_model, dm, cfg)

# 2. Run Simple Split (Using your pre-loaded simple_failures DataFrame)
evaluator.run_logit_lens(simple_failures, split_name="Simple", sample_n=None)
evaluator.print_metrics()
evaluator.inspect_sample(idx=0, only_late_failures=True)

# 3. Run Length Split
evaluator.run_logit_lens(length_failures, split_name="Length", sample_n=None)
evaluator.print_metrics()

# 4. Run Add Primitive Jump Split
evaluator.run_logit_lens(addprim_jump_failures, split_name="Add Prim Jump", sample_n=None)
evaluator.print_metrics()

Running dynamic logit lens on 2764 examples from 'Simple' split...

=================== SCAN SIMPLE SPLIT METRICS ===================
Early Failure (Layer 1 Dissolution)   : 1693 cases (61.3%)
Late Failure (Layer 2 Overwrite)      : 1071 cases (38.7%)
-----------------------------------------------------------------
Average generation step where failure begins: 9.19

Running dynamic logit lens on 2659 examples from 'Length' split...

=================== SCAN LENGTH SPLIT METRICS ===================
Early Failure (Layer 1 Dissolution)   : 1471 cases (55.3%)
Late Failure (Layer 2 Overwrite)      : 1187 cases (44.7%)
-----------------------------------------------------------------
Average generation step where failure begins: 18.50

Running dynamic logit lens on 4730 examples from 'Add Prim Jump' split...

=================== SCAN ADD PRIM JUMP SPLIT METRICS ===================
Early Failure (Layer 1 Dissolution)   : 2886 cases (61.0%)
Late Failure (Layer 2 Overwrite)      : 1843 cases (

## 3. Results & Discussion

### 3.1 Simple Split

* Early-Stage State Dissolution (Layer 1 Dissolution): 1,693 cases (**61.3%**)
> The predominant failure mode operates via a breakdown in the initial half of the computational pipeline. At the specific execution step $t$, the correct next-token projection fails to materialize at the boundary of Layer 1. This implies that the model's foundational features—localized in Layer 0 attention blocks or the subsequent MLP layers—are failing to preserve structural context or state tracking over long horizons. By the time information enters the final layer, the contextual representations have already dissolved into an unrecoverable latent state.

* Late-Stage Structural Overwrite (Layer 2 Overwrite): 1,071 cases (**38.7%**)
> This confirms a failure of compositionality rather than a failure of representation. Layer 1 computes the correct grammatical operation, but circuits within Layer 2 introduce a destructive intervention. This is frequently driven by strong, un-contextualized bigram biases or highly localized attention sinks that override the foundational hidden states right before the final unembedding projection (`ln_final` $\rightarrow$ `unembed`).

* Mean Temporal Error Onset ($\tau_{err}$): **9.19 generation steps**
> The temporal metric indicates that the model is robust during the execution of brief or highly localized command strings. The degradation of hidden states is strongly correlated with sequence length and generation depth, implicating a compounding informational entropy or a structural failure in tracking long-range autoregressive dependencies.

---

### 3.2 Length Split

* Early-Stage State Dissolution (Layer 1 Dissolution): 1,471 cases (**55.3%**)
> Compared to the distribution-matched splits, early-stage dissolution decreases slightly here. This suggests that Layer 1 remains relatively adept at encoding the basic compositional directives even when confronted with longer command sequences. However, more than half of the failures are still born from early-stage representation decay, signaling that the lower layers are fundamentally constrained in scale-invariant tracking when sequence contexts expand past training thresholds.

* Late-Stage Structural Overwrite (Layer 2 Overwrite): 1,187 cases (**44.7%**)
> Notably, the Length Split exhibits the highest propensity for late-stage overwrites across all test distributions. This reveals that lower-level representations frequently calculate the correct compositional trajectory, but Layer 2 suffers severe structural out-of-distribution (OOD) breakdown. Because the final layer cannot reconcile the unprecedented geometric lengths with its rigid attention-sink mechanics, it aggressively forces a collapse into overlearned bigram boundaries or premature terminations.

* Mean Temporal Error Onset ($\tau_{err}$): **18.50 generation steps**
> The drastic elevation in the temporal error onset to 18.50 steps provides clear empirical evidence of the model's basic generative stamina. The system does not immediately lose its structural coherence upon facing an OOD sequence. Instead, it successfully generalizes the programmatic grammar for a considerable horizon before hitting a systemic memory ceiling, confirming that state degradation is an accumulated, step-dependent function of autoregressive rollout depth.

---

### 3.3 Add Prim Jump Split

* Early-Stage State Dissolution (Layer 1 Dissolution): 2,886 cases (**61.0%**)
> When integrating a newly isolated primitive component (`JUMP`), early dissolution mirrors the base error profile of the Simple Split. This indicates that the introduction of a novel structural primitive triggers an identical failure of representations in Layer 1. The lower layers face a systematic bottleneck where the primitive feature mapping fails to remain coherent when embedded inside complex modifier patterns, causing early state collapse.

* Late-Stage Structural Overwrite (Layer 2 Overwrite): 1,843 cases (**39.0%**)
> The late overwrite behavior persists strongly when contextualizing the primitive element. Even when Layer 1 effectively maps the new primitive into the syntax and prepares to pass the correct target token upward, Layer 2’s established attention circuits act destructively. It forcefully maps the novel composition onto historic, heavily weight-biased primitive templates (such as `WALK` or `RUN` configurations), over-riding the lower layer's accurate computations.

* Mean Temporal Error Onset ($\tau_{err}$): **9.64 generation steps**
> The alignment of the temporal error onset (9.64) with the Simple Split suggests that local primitive composition errors share a uniform computational overhead. The model safely navigates the syntax initialization phase, but begins to break down at roughly the same structural depth. This reinforces that primitive isolation failures are bound to the exact same tracking limitations as standard execution phrases.

## 4. Next Steps: Attention Circuit Isolation

To transition from architectural localization (layers) to discrete functional circuits, the subsequent notebook (`02_attention_patterns.ipynb`) will isolate specific mathematical components driving these two failure modes.

### 4.1 Characterizing Early-Stage Dissolution ($t > 8$)
We will map the attention pattern matrices of **Layer 0** during extended generation trajectories. 
* **Hypothesis:** The early attention heads fail to adequately distribute attention weights back to long-range context markers in the input prompt (e.g., structural modifiers such as `after`, `thrice`, or `opposite`), precipitating the immediate state dissolution observed at step 9.19.

### 4.2 Isolating Destructive Sub-Circuits in Layer 1
We will analyze the direct logit attribution of individual attention heads in **Layer 1** for the 38.7% of cases flagged as late overwrites.
* **Hypothesis:** There exists a subset of over-generalized execution or bigram heads within the final layer that heavily prioritize the immediate local history ($t-1$), effectively blinding the final unembedding layer to the programmatic routing information passed up through Layer 1's residual stream